In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("brfss_clean_2020_2024.csv")

race_map = {
    1: "NH-White", 2: "NH-Black", 3: "AIAN",
    4: "Asian", 5: "NHOPI", 6: "Other/Multiracial",
    7: "Hispanic"
}
age_map = {
    1: "18-24", 2: "25-29", 3: "30-34", 4: "35-39",
    5: "40-44", 6: "45-49", 7: "50-54", 8: "55-59",
    9: "60-64", 10: "65-69", 11: "70-74", 12: "75-79", 13: "80+"
}
sex_map = {1: "Male", 2: "Female"}
education_map = {
    1: "Did not graduate high school",
    2: "Graduated high school",
    3: "Attended college or technical school",
    4: "Graduated college or technical school"
}
income_map = {
    1: "<15k", 2: "15k-25k", 3: "25k-35k",
    4: "35k-50k", 5: "50k-100k", 6: "100k-200k", 7: "200k+"
}

df["race_group"]   = df["_RACEPRV"].map(race_map)
df["age_group"]    = df["_AGEG5YR"].map(age_map)
df["sex"]          = df["_SEX"].map(sex_map)
df["education"]    = df["_EDUCAG"].map(education_map)
df["income_group"] = df["_INCOMG1"].map(income_map)

df_model = df.dropna(subset=["race_group", "age_group", "sex",
                               "education", "income_group", "obese"]).copy()
df_model = df_model.reset_index(drop=True)

le_state = LabelEncoder()
df_model["state_code"] = le_state.fit_transform(df_model["_STATE"])

print("Shape:", df_model.shape)
print("Model dataset ready.")

Shape: (1322240, 20)
Model dataset ready.


In [3]:
# Base features
df_encoded = pd.get_dummies(df_model[["age_group", "sex", "education",
                                       "income_group", "race_group"]], drop_first=True)
state_dummies = pd.get_dummies(df_model["state_code"], prefix="state", drop_first=True)

# Interaction terms using pandas multiplication
age_dummies = pd.get_dummies(df_model["age_group"], prefix="age", drop_first=True)
race_dummies = pd.get_dummies(df_model["race_group"], prefix="race", drop_first=True)
income_dummies = pd.get_dummies(df_model["income_group"], prefix="inc", drop_first=True)
edu_dummies = pd.get_dummies(df_model["education"], prefix="edu", drop_first=True)

# Age x Race interactions
age_race_cols = {}
for a in age_dummies.columns:
    for r in race_dummies.columns:
        age_race_cols[f"{a}_x_{r}"] = (age_dummies[a] * race_dummies[r]).values
age_race = pd.DataFrame(age_race_cols)

# Income x Race interactions
inc_race_cols = {}
for i in income_dummies.columns:
    for r in race_dummies.columns:
        inc_race_cols[f"{i}_x_{r}"] = (income_dummies[i] * race_dummies[r]).values
inc_race = pd.DataFrame(inc_race_cols)

# Education x Race interactions
edu_race_cols = {}
for e in edu_dummies.columns:
    for r in race_dummies.columns:
        edu_race_cols[f"{e}_x_{r}"] = (edu_dummies[e] * race_dummies[r]).values
edu_race = pd.DataFrame(edu_race_cols)

# Combine all features
X_interaction = pd.concat([
    df_encoded,
    state_dummies,
    age_race,
    inc_race,
    edu_race
], axis=1)

y = df_model["obese"].values
w = df_model["_LLCPWT_adjusted"].values
race_all = df_model["race_group"].values

print(f"Base features: {df_encoded.shape[1] + state_dummies.shape[1]}")
print(f"Age x Race: {age_race.shape[1]}")
print(f"Income x Race: {inc_race.shape[1]}")
print(f"Education x Race: {edu_race.shape[1]}")
print(f"Total features: {X_interaction.shape[1]}")

# Train test split
X_train, X_test, y_train, y_test, w_train, w_test, race_train, race_test = train_test_split(
    X_interaction, y, w, race_all,
    test_size=0.2, random_state=42, stratify=y
)

print("\nFitting interaction model...")
lr_int = LogisticRegression(max_iter=1000, solver="lbfgs", C=0.1)
lr_int.fit(X_train, y_train, sample_weight=w_train)
print("Done.")

Base features: 81
Age x Race: 72
Income x Race: 36
Education x Race: 18
Total features: 207

Fitting interaction model...
Done.


In [4]:
# Predictions
y_pred_int = lr_int.predict_proba(X_test)[:, 1]

# Overall AUC
overall_auc_int = roc_auc_score(y_test, y_pred_int, sample_weight=w_test)
print(f"Overall AUC — baseline: 0.6354")
print(f"Overall AUC — interactions: {overall_auc_int:.4f}")

# Race-specific AUC comparison
print("\n=== RACE-SPECIFIC AUC COMPARISON ===")
print(f"{'Race Group':25s} {'Baseline':>10} {'Interactions':>13} {'Change':>8}")
print("-" * 60)

baseline_aucs = {
    "AIAN": 0.6100, "Asian": 0.6013, "Hispanic": 0.5883,
    "NH-Black": 0.5963, "NH-White": 0.6204,
    "NHOPI": 0.5448, "Other/Multiracial": 0.6150
}

interaction_results = []
for race in sorted(pd.Series(race_test).unique()):
    mask = race_test == race
    y_race = y_test[mask]
    pred_race = y_pred_int[mask]
    w_race = w_test[mask]

    if len(np.unique(y_race)) < 2:
        continue

    auc_int = roc_auc_score(y_race, pred_race, sample_weight=w_race)
    auc_base = baseline_aucs.get(race, 0)
    change = auc_int - auc_base
    flag = "⬆️" if change > 0.005 else "⬇️" if change < -0.005 else "➡️"

    interaction_results.append({
        "race_group": race,
        "baseline_auc": auc_base,
        "interaction_auc": round(auc_int, 4),
        "change": round(change, 4)
    })

    print(f"{race:25s} {auc_base:>10.4f} {auc_int:>13.4f} {change:>+8.4f} {flag}")

results_df = pd.DataFrame(interaction_results)
print("\nGroups still below 0.60:")
print(results_df[results_df["interaction_auc"] < 0.60][
    ["race_group", "baseline_auc", "interaction_auc", "change"]].to_string(index=False))

Overall AUC — baseline: 0.6354
Overall AUC — interactions: 0.6369

=== RACE-SPECIFIC AUC COMPARISON ===
Race Group                  Baseline  Interactions   Change
------------------------------------------------------------
AIAN                          0.6100        0.6055  -0.0045 ➡️
Asian                         0.6013        0.6156  +0.0143 ⬆️
Hispanic                      0.5883        0.5905  +0.0022 ➡️
NH-Black                      0.5963        0.6008  +0.0045 ➡️
NH-White                      0.6204        0.6213  +0.0009 ➡️
NHOPI                         0.5448        0.5552  +0.0104 ⬆️
Other/Multiracial             0.6150        0.6096  -0.0054 ⬇️

Groups still below 0.60:
race_group  baseline_auc  interaction_auc  change
  Hispanic        0.5883           0.5905  0.0022
     NHOPI        0.5448           0.5552  0.0104


## Interaction Model Analysis

### Motivation
Three racial groups showed AUC below the 0.60 suppression threshold in the baseline
logistic regression — NHOPI (0.545), Hispanic (0.588), and NH-Black (0.596). The baseline
model uses only main effects, meaning it assumes the relationship between demographics
and obesity is the same across all racial groups. This is unlikely to be true — the income
gradient for Hispanic respondents may differ from NH-White, and the age trajectory for
NH-Black may peak earlier than other groups.

Interaction terms allow the model to learn group-specific demographic relationships
without changing the cell structure or requiring separate models.

### Interactions Added
Three sets of interaction terms added to the baseline 81 features:

| Interaction | Features Added | Rationale |
|-------------|---------------|-----------|
| Age × Race | 72 | Obesity-age trajectory differs by race |
| Income × Race | 36 | Income gradient varies across racial groups |
| Education × Race | 18 | Education-obesity relationship differs by group |

Total features: 207 (vs 81 baseline)

### Results

| Race Group | Baseline AUC | Interaction AUC | Change |
|------------|-------------|-----------------|--------|
| NH-White | 0.6204 | 0.6213 | +0.0009 ➡️ |
| Other/Multiracial | 0.6150 | 0.6096 | -0.0054 ⬇️ |
| AIAN | 0.6100 | 0.6055 | -0.0045 ➡️ |
| Asian | 0.6013 | 0.6156 | +0.0143 ⬆️ |
| NH-Black | 0.5963 | 0.6008 | +0.0045 ⬆️ |
| Hispanic | 0.5883 | 0.5905 | +0.0022 ➡️ |
| NHOPI | 0.5448 | 0.5552 | +0.0104 ⬆️ |

Overall AUC: 0.6354 → 0.6369 (+0.0015)

### Key Findings

**Interactions helped but did not fully resolve the low AUC problem.**
Asian improved meaningfully (+0.014) and NH-Black crossed the 0.60 threshold (+0.005).
However Hispanic (0.591) and NHOPI (0.555) remain below threshold despite interactions.

**Hispanic remains the most concerning group.**
With 24,390 test observations, sample size is not the issue. Even with race-specific
interaction terms, demographics alone cannot adequately explain within-group obesity
variation for Hispanic respondents. Cultural, geographic, and community-level factors
not captured in BRFSS demographics are likely driving within-group variation.

**NHOPI improvement (+0.010) is encouraging but sample size remains a fundamental limit.**
Only 1,304 NHOPI test observations — insufficient for interaction terms to be reliably
estimated. More data is the only real fix for NHOPI.

### Why This Model Was Not Used to Update the Group Summary
The overall AUC improvement was only 0.0015 — too small to justify replacing the
existing modeled obesity rates that the ACS team and modeling team are already working with.
The correlation between baseline and interaction model cell predictions would be extremely
high (>0.99), making the update negligible in practice.

Instead race-specific models are being built for Hispanic and NHOPI — the two groups
that remain below the 0.60 threshold — to see if within-group modeling produces
meaningful AUC improvements for those specific populations.